# Intrusion Detection Using Ensemble Techniques
Run the whole project in your browser. In Colab: **Runtime > Run all**. Training takes about 2 minutes.

In [ ]:
!git clone -q https://github.com/aish-19shanbhag/intrusion-detection-ensemble.git
%cd intrusion-detection-ensemble
!pip install -q -r requirements.txt

## 1. The data
NSL-KDD: 125,973 training connections and 22,544 test connections, 41 features each, grouped into five categories.

In [ ]:
from src.config import TRAIN_FILE, TEST_FILE
from src.data_loader import load_nsl_kdd
train, test = load_nsl_kdd(TRAIN_FILE), load_nsl_kdd(TEST_FILE)
print(train.shape, test.shape)
train['attack_cat'].value_counts()

## 2. Feature selection and tuning (optional, about 5 minutes)
The tuned parameters are already saved in `reports/metrics/best_params.json`. Uncomment to re-run.

In [ ]:
# !python -m src.tune

## 3. Train, benchmark and evaluate

In [ ]:
!python -m src.train

## 4. Results

In [ ]:
import json, pandas as pd
m = json.load(open('reports/metrics/metrics.json'))
for section in ['holdout', 'official_benchmark']:
    rows = {k: {x: v[x] for x in ['accuracy', 'macro_f1', 'false_positive_rate', 'detection_rate']} for k, v in m[section]['results'].items()}
    print(section); display(pd.DataFrame(rows).T.style.format('{:.4%}'))

In [ ]:
from IPython.display import Image, display
for f in ['model_comparison_holdout', 'confusion_matrix_holdout', 'model_comparison_official', 'per_class_recall_official']:
    display(Image(f'reports/figures/{f}.png'))

## 5. Score new connections

In [ ]:
import pandas as pd
from src.predict import predict_frame
sample = pd.read_csv('data/sample_connections.csv')
result = predict_frame(sample)
result.insert(0, 'actual', sample['attack_cat'])
result